# $Bx$ component as Ito's process 
This notebook is devoted to analysis of connection between $a(t)$ and $b(t)$ 
from Ito equation: 
$$dX = a(t)dt + b(t)dW, \quad \text{where } W \text { is a normal Wiener process}$$

In [ ]:
# Import modules
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode
from IPython.display import display, HTML
import pickle

# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2
# Enable latex on plotly figures
init_notebook_mode()
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

## Calculate $a(t), \space b(t)$

In [ ]:
# Load up gaussian mixture model for component Bx
with open("gmmBx.pkl", "rb") as f:
    gmm = pickle.load(f)

Parameters of Ito's equation are calculated in the next way:

$$
a(t) = \sum_{k=1}^{K}{p_k a_k}, \quad 
b(t) = \sum_{k=1}^{K}{p_k b_k},
$$
where $K$ is a number of mixture components

In [ ]:
p = gmm["weights"]
a = gmm["means"]
b = gmm["variances"]

coef_a = np.sum(p * a, axis=0)
coef_b = np.sum(p * b, axis=0)

# Correlation plots

In [ ]:
def correlation_plot(correlation, title):
    fig = make_subplots(rows=3, cols=1)
    # Dividing plot on three parts
    part1, part2, part3 = [], [], []
    for i, val in enumerate(correlation):
        if (i + 1) / len(correlation) < 1 / 3:
            part1.append(val)
            i1 = i
        elif (i + 1) / len(correlation) < 2 / 3:
            part2.append(val)
            i2 = i
        else:
            part3.append(val)
    # Add plots to figure
    fig.append_trace(
        go.Scatter(
            x=gmm["dates"][:i1],
            y=part1,
            name="First part",
        ),
        row=1,
        col=1,
    )

    fig.append_trace(
        go.Scatter(
            x=gmm["dates"][i1:i2],
            y=part2,
            name="Second part",
        ),
        row=2,
        col=1,
    )

    fig.append_trace(
        go.Scatter(
            x=gmm["dates"][i2:],
            y=part3,
            name="Third part",
        ),
        row=3,
        col=1,
    )

    fig.update_layout(height=800, width=1200, title_text=title)
    return fig

In [ ]:
correlation_window = (60 * 12, 1)
kernel_size = 60 * 4  # For smoothing plots

#### Correlation $a(t)$, $b(t)=\sum_{j=1}^Kp_jb_j$

In [ ]:
correlation = []
for i in range(0, len(coef_a) - correlation_window[0], correlation_window[1]):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = coef_b[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
title = (
    r"""
$\text{Correlation between } a(t), b(t)=\sum_{j=1}^Kp_jb_j \text{ on window size }
"""
    + rf"{correlation_window[0]} \text{{ minutes}}$"
)
cp = correlation_plot(correlation, title)
cp.show()
cp.write_image(f"/tmp/corr3_{correlation_window[0]}.png")

#### Correlation $a(t)$, $b^2(t)$

In [ ]:
correlation2 = []
for i in range(0, len(coef_a) - correlation_window[0], correlation_window[1]):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = (coef_b**2)[i : correlation_window[0] + i]
    correlation2.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
title = (
    r"""
$\text{Correlation between } a(t), b^2(t)=\left(\sum_{j=1}^Kp_jb_j\right)^2 \text{ on window size }
"""
    + rf"{correlation_window[0]} \text{{ minutes}}$"
)
cp = correlation_plot(correlation2, title)
cp.show()
cp.write_image(f"/tmp/corr4_{correlation_window[0]}.png")

# Analysis of $a(t)$ and $b(t)$ relation

## Trigonometric approximation

In [ ]:
def harmonic_approximation(data, time, harmonics_num=4, title=""):
    import numpy as np
    from scipy.optimize import leastsq
    import plotly.graph_objects as go
    from scipy.stats import kstest, norm
    from plotly.subplots import make_subplots

    def find_frequency(data, sampling_rate):
        n = len(data)
        fft_result = np.fft.fft(data)
        freqs = np.fft.fftfreq(n, d=1 / sampling_rate)
        spectrum = abs(fft_result)

        idx = np.argmax(spectrum[1:]) + 1  # Избегаем нулевую частоту
        freq = freqs[idx]

        return abs(freq)

    fig = make_subplots(rows=3, cols=1)
    # fig = make_subplots(rows=2, cols=1)
    data_orig = data
    t = np.array(range(len(data)))
    params = []
    harmonical_signal = np.zeros(len(data))

    for _ in range(harmonics_num):
        guess_mean = np.mean(data)
        guess_phase = 0
        guess_freq = find_frequency(data, len(data)) * np.pi * 2 / len(data)
        guess_amp = max(data) - min(data)

        def optimize_func(x):
            return x[0] * np.sin(x[1] * t + x[2]) + x[3] - data

        params_sin = leastsq(
            optimize_func, [guess_amp, guess_freq, guess_phase, guess_mean]
        )[0]

        params.append(params_sin)
        est_amp, est_freq, est_phase, est_mean = params_sin

        data_fit = est_amp * np.sin(est_freq * t + est_phase) + est_mean
        data = data - data_fit
        harmonical_signal += data_fit

    # Approximation visualization
    fig.add_trace(
        go.Scatter(x=time, y=data_orig, marker=dict(color="#3058B0")),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=time, y=harmonical_signal, marker=dict(color="#FF7F50")),
        row=1,
        col=1,
    )
    fig.add_annotation(
        xref="x domain",
        yref="y domain",
        x=0.5,
        y=1.2,
        showarrow=False,
        font=dict(size=22),
        text=f"<b>Approximation ({harmonics_num} harmonics)<b>",
        row=1,
        col=1,
    )
    # Residuals
    fig.add_trace(go.Scatter(x=time, y=data, marker=dict(color="purple")), row=2, col=1)
    fig.add_annotation(
        xref="x domain",
        yref="y domain",
        x=0.5,
        y=1.2,
        showarrow=False,
        font=dict(size=22),
        text="<b>Residuals<b>",
        row=2,
        col=1,
    )
    # Histogramm
    fig.add_trace(
        go.Histogram(
            x=data, histnorm="probability density", marker=dict(color="#3058B0")
        ),
        row=3,
        col=1,
    )
    x = np.linspace(min(data) * 1.2, max(data) * 1.2, 100)
    fig.add_trace(
        go.Scatter(x=x, y=norm.pdf(x, *norm.fit(data)), marker=dict(color="#FF7F50")),
        row=3,
        col=1,
    )

    def cdf(x):
        return norm.cdf(x, loc=norm.fit(data)[0], scale=norm.fit(data)[1])

    pval = kstest(data, cdf=cdf).pvalue
    fig.add_annotation(
        xref="x domain",
        yref="y domain",
        x=0.5,
        y=1.3,
        showarrow=False,
        font=dict(size=22),
        text="<b>Residuals histogram<b>",
        row=3,
        col=1,
    )
    fig.add_annotation(
        xref="x domain",
        yref="y domain",
        x=0.5,
        y=1.165,
        showarrow=False,
        font=dict(size=20),
        text=f"Kolmogorov-Smirnov p-value = {pval:.3f}",
        row=3,
        col=1,
    )

    fig.update_layout(
        width=1200,
        height=1000,
        title=dict(font=dict(size=20), text=f"<b>{title}<b>"),
        showlegend=False,
    )
    return fig, data, params

In [ ]:
fig, *_ = harmonic_approximation(
    data=correlation,
    time=gmm["dates"][: len(correlation)],
    harmonics_num=12,
    title="Bx - correlation a(t), b(t)",
)
fig.show()

## Dynamic linear regression